[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Reinforcement_Learning.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Reinforcement Learning

Learning from *consequences* instead of labels — the paradigm the [LLM workshop](./LLMs_from_the_Ground_Up.ipynb) name-dropped as RLHF and this course delivers. Five sessions: bandits, MDPs & Bellman, temporal-difference learning, policy gradients, and the road to RLHF — every algorithm verified against an exactly-solvable environment.

## 1. Pre-requisites

- [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb) & [Independence](../Intro_Math/Analysis/Independence.ipynb) (expectations, LLN).
- [Training Dynamics](./Training_Dynamics.ipynb) for Session 4.
- Kinship worth knowing: TD learning's *new = old + α·(surprise)* is the [adaptive-filter heartbeat](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) yet again.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 5 — *Bandits: Exploration vs Exploitation* (~35 min)
**Goal:** the RL problem with no states: regret, ε-greedy, and UCB's optimism.
**Feeds into:** Session 2 (MDPs).

---

## 2. The Ten-Armed Testbed

💡 **Intuition.** Ten slot machines, unknown payouts, 1000 pulls: every pull spent *learning* is a pull not spent *earning*. That tension — exploration vs exploitation — is RL's signature dilemma, isolated from everything else. **ε-greedy** explores blindly and forever; **UCB** explores *strategically*: pull the arm whose plausible upside $\hat\mu_a + c\sqrt{\ln t / n_a}$ is highest — 'optimism in the face of uncertainty', with the bonus shrinking exactly like a [confidence interval](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb).

In [ ]:

# YOUR CODE HERE


**What just happened.** Three one-line policies, 1000 pulls, averaged over 800 random bandits — and three **qualitatively different curve shapes**, which is more informative than the totals in the legend.

- **Greedy** rises fast, then goes **flat** — and flat is bad news here, because a flat cumulative-regret curve at a nonzero slope means it has stopped improving. It locked onto whichever arm looked good after one pull and never checked another. With 10 arms it lands on the true best about 1 time in 10 and pays the gap forever the rest of the time.
- **ε-greedy** is a **straight line**. It keeps learning, but it also keeps paying: 10% of every pull is spent on a uniformly random arm, *including long after it knows the answer*. Constant tax, linear regret.
- **UCB** bends over and keeps bending. Its exploration cost **shrinks with evidence**.

**Those shapes are asymptotic classes, not tuning differences.** Greedy and ε-greedy both have **linear** regret; UCB has **logarithmic** regret and is provably optimal up to constants. **No fixed ε achieves a logarithmic rate** — you would need a decaying schedule, and even then tuning it is exactly the problem UCB solves automatically.

**The mechanism is in the bonus term, and it is a confidence interval wearing a different hat.** $\hat\mu_a + c\sqrt{\ln t / n_a}$ says "my estimate, plus how wrong I could plausibly be". The $1/\sqrt{n_a}$ is the standard error of a sample mean; the $\ln t$ widens it enough to hold simultaneously across all times. **Optimism in the face of uncertainty**: an arm with few pulls has a large bonus, so it gets pulled; the bonus then shrinks on its own. **Exploration is scheduled by the data, not by a hyperparameter** — which is precisely what ε-greedy cannot do.

**Note the line doing the learning, because it will reappear three more times in this workshop.** `Q[a] += (reward - Q[a]) / N[a]` is an incremental mean, and it has the shape **new = old + α × (surprise)**. That is the LMS update from [Adaptive Filtering](../Intro_Time_Series/Intro_AdFilt_APA.ipynb), it becomes the TD update in Session 3, and it is the generic form of stochastic approximation. **One line, four workshops.**

**Two honest caveats before generalising the ranking.** First, $c = 2.0$ is a *choice*: UCB's optimality is about the **rate**, and the constant still needs setting — a poorly tuned UCB can lose to a well-tuned ε-greedy over 1000 pulls. Second, the arms here are stationary Gaussians with known noise scale. **On non-stationary problems UCB's shrinking bonus is a liability**, because it stops exploring an arm whose payoff has since changed, and that is a real failure mode in recommender systems.

**Finally, keep the framing in view: this is RL with everything removed except one dilemma.** No states, no transitions, no delayed reward, no credit assignment. **Every pull spent learning is a pull not spent earning**, and that tension survives into every later session — it is just harder to see once states and time are added back.

---
### 🕐 Session 2 of 5 — *MDPs & the Bellman Equation* (~40 min)
**Goal:** add states and time; solve a gridworld EXACTLY by dynamic programming.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (TD learning).

---

## 3. Markov Decision Processes

💡 **Intuition.** Now actions have *consequences that persist*: an MDP is states, actions, transition probabilities, rewards, and a discount $\gamma$. The value $V^\pi(s)$ is expected discounted return — and Bellman's equation says value is **recursively self-consistent**: today's value = today's reward + γ·tomorrow's value. The optimal version ($V^* = \max_a [r + \gamma E V^*]$) is a fixed-point equation, and **value iteration** just applies it until it stops moving — a contraction ([Sequences & Series](../Intro_Math/Analysis/Numerical_Sequences_and_Series.ipynb): Cauchy convergence with rate γ!). This gives us an *exact oracle* to test every learning algorithm against.

In [ ]:
# 4x4 gridworld: start anywhere, goal at (3,3) reward +1, pit at (1,2) reward −1, step −0.02
# value iteration = the exact solution (our ORACLE for everything later)

# YOUR CODE HERE


**What just happened.** Value iteration converged in **32 sweeps** and produced a policy whose arrows all point, eventually, toward the goal at $(3,3)$ — while routing *around* the pit at $(1,2)$ rather than past it.

**The policy is checkable, so check it rather than admiring it.** Look at column 3: states $(0,3)$, $(1,3)$, $(2,3)$ all say **↓**, walking straight down the right edge to the goal. Now look at $(0,2)$, directly above the pit: it says **→**, stepping sideways into the safe column instead of down. **The 10% slip probability is what makes that detour worth −0.02 per step** — a direct route past the pit risks a −1.0 with probability 0.1, and the arithmetic prefers the longer path. Remove the slip and the policy changes.

**Now the number that does not match the naive bound, and it is worth stopping on.** A $\gamma$-contraction should shrink the residual by $0.95$ per sweep, so reaching $10^{-12}$ would need about **540 sweeps**. It took **32**, implying an effective per-sweep factor near $0.42$. **The $\gamma$ bound is worst-case and this MDP beats it**, because the goal and pit are **absorbing**: once an episode terminates no value flows back, so the true contraction is $\gamma \times P(\text{not yet terminated})$. Episodes here end within a handful of steps, and the convergence rate reflects that.

**That is the general lesson about contraction bounds, not a quirk of this grid.** $\gamma$ guarantees convergence and bounds the *worst* case; the actual rate depends on the structure of the problem. **A method converging faster than its bound is normal; converging slower would be a bug.**

**Note where the discount factor is doing load-bearing work.** $\gamma = 0.95$ is what makes the Bellman operator a contraction and therefore makes the fixed point **unique and reachable**. It is not a statement about preferring near-term reward — it is what makes the problem well-posed. Set $\gamma = 1$ with no terminal state and values can diverge; the infinite sum need not converge at all.

**And note what value iteration required, because Session 3 is defined by not having it.** Every transition probability, every reward, for every state–action pair — `step_model` hands them over in closed form. **No agent in the world has that.** A robot does not know the probability that its wheel slips; a trading system does not know the market's transition kernel.

**Which is exactly why this $4\times4$ grid is the right size.** Because we own the model, we own $V^*$, $Q^*$, and $\pi^*$ **exactly** — a genuine answer key. Every learning algorithm in the remaining sessions is graded against it, including the value $V^*(\text{start}) = 0.642$ that the policy-gradient plot uses as its reference line. **Almost no RL tutorial can check its own answers**, and choosing a solvable environment over an impressive one is what makes that possible here.

---
### 🕐 Session 3 of 5 — *Temporal-Difference Learning* (~40 min)
**Goal:** learn the same values WITHOUT the model: TD(0) and Q-learning, checked against the oracle.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (policy gradients).

---

## 4. Learning from the Surprise

💡 **Intuition.** Value iteration needed the transition model. An *agent* only gets experience: $(s, a, r, s')$. TD's move: use the Bellman equation as an **error signal** — the *TD error* $\delta = r + \gamma \max_a Q(s', a) - Q(s, a)$ is how surprised you are, and $Q \mathrel{+}= \alpha \delta$ is the [LMS update](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) with the Bellman backup as the 'desired signal'. Q-learning does this off-policy (learns the greedy value while exploring) — and, on a finite MDP with decaying exploration, provably converges to $Q^*$. We *check* that, since we own the oracle.

In [ ]:
# tie-aware policy check: the learned greedy action must be (near-)optimal under Q*

# YOUR CODE HERE


**What just happened.** Q-learning, from experience alone with **no access to the transition model**, produced:

- $\|Q - Q^*\|_\infty = \mathbf{0.232}$
- learned greedy action optimal in **100%** of states

**Those two numbers disagree, and the disagreement is the most useful thing in the session.** On a scale where the true values run from 0 to 0.99, an error of 0.232 is **not small** — roughly a quarter of the range. Yet every single state's greedy action matches the oracle. **How can the values be that wrong and the policy be perfect?**

**Because a policy only needs the *argmax*, not the magnitudes.** If the best action at a state beats the runner-up by 0.3, then both estimates can be off by 0.14 without changing the decision. **Value accuracy and policy optimality are different objectives**, and control cares about the second. That is why the notebook's check is tie-aware — comparing $Q^*[s, \hat a]$ against $\max_a Q^*[s,a]$ rather than comparing $Q$ against $Q^*$ elementwise.

**Ask where the 0.232 actually lives, because the answer is diagnostic.** It sits in **rarely-visited state–action pairs** — above all, actions that step toward the pit. The ε-greedy policy avoids them once it has learned they are bad, so their counts stay low, their step sizes stay large, and their estimates stay noisy. **Q-learning converges where it looks**, and the residual is a map of where it stopped looking. This is Session 1's exploration/exploitation tension, still present with states attached.

**The convergence itself is not luck — the code satisfies a theorem's hypotheses on purpose.** `alpha = 1.0/N_sa[s,a]**0.6` gives $\sum\alpha = \infty$ (enough total movement to reach any value) and $\sum\alpha^2 < \infty$ (enough decay to stop bouncing) — the **Robbins–Monro** conditions. Most tutorials use a constant $\alpha$ and thereby forfeit any guarantee. **Convergence here is a theorem with hypotheses, and the exponent 0.6 is how they are met.**

**Note the update rule one more time, because it is now unmistakable.** `Q[s,a] += alpha * (target - Q[s,a])` is **new = old + α × surprise** — the LMS update from [Adaptive Filtering](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) with the Bellman backup as the desired signal, identical in shape to Session 1's incremental mean and to stochastic gradient descent. **Four workshops, one line.**

**And note the word "max" in the target, which is what makes this Q-learning rather than SARSA.** The update uses $\max_a Q(s',a)$ — the value of the **greedy** action — while behaviour is ε-greedy and frequently random. **You learn about the optimal policy while following a different one.** That off-policy property is what makes replay buffers, offline RL, and learning from logged human data possible at all.

**Finally, the boundary this result does not cross.** The convergence theorem holds because the table has one independent entry per state–action pair — 64 numbers, each updated by its own average. **Replace the table with a neural network and the guarantee disappears.** Function approximation, bootstrapping, and off-policy updates together form the "deadly triad", and their combination can diverge. Target networks and replay buffers in DQN are engineering patches for exactly the theorem that this tabular setting gets for free.

---
### 🕐 Session 4 of 5 — *Policy Gradients* (~40 min)
**Goal:** skip values, optimize the policy directly: REINFORCE with a baseline, from scratch.
**Builds on:** Session 3; [Training Dynamics](./Training_Dynamics.ipynb). &nbsp; **Feeds into:** Session 5 (the road to RLHF).

---

## 5. Differentiating Through Luck

💡 **Intuition.** Values are a detour; why not adjust the policy's parameters to make good episodes more likely? The log-derivative trick makes the un-differentiable differentiable: $\nabla E[R] = E[R \, \nabla \log \pi(a|s)]$ — *reinforce the log-probability of what you did, in proportion to how well it went*. The estimator is unbiased but wildly noisy ([SGD's](../Intro_Math/Optimization/Optimization.ipynb) noise-floor problem, squared); subtracting a **baseline** (the mean return) cancels variance without adding bias — the single most important practical trick in policy-land.

In [ ]:
# REINFORCE on the same gridworld (tabular softmax policy)

# YOUR CODE HERE


**What just happened.** Two REINFORCE runs, both climbing to the dotted line at $V^*(\text{start}) = \mathbf{0.64}$ — the value that Session 2's dynamic programming computed **exactly**, without any learning. A policy-gradient method that never estimated a value function, never used the transition model, and only ever sampled episodes, arrived at the optimum.

**Read the right feature of the plot: the difference is *width*, not height.** Both curves reach roughly the same place. **The baseline does not find a better policy — it finds the same one more calmly.** Compare the raggedness of the two traces; that is the variance reduction, and it is the whole content of the comparison. Reading the endpoints instead of the smoothness is the easy mistake.

**Why the baseline works is one line of algebra, and it is worth doing because "subtract the mean" sounds arbitrary.**

$$E[b\,\nabla\log\pi] = b\,\nabla\!\!\int\!\pi\,da = b\,\nabla 1 = 0$$

**Any baseline that does not depend on the action leaves the gradient exactly unbiased.** So variance falls and nothing is paid for it. That is unusual — most variance reductions cost bias — and it is why the baseline is the single most important practical trick in policy-gradient methods.

**The intuition is sharper than the algebra.** Without a baseline, *every* action taken during a successful episode gets reinforced, including the bad ones that merely co-occurred with success. With a baseline, only **better-than-average** actions are pushed up and worse-than-average ones are pushed down. **The learning signal changes from "was this good?" to "was this better than usual?"**

**Note what made the whole method possible: the log-derivative trick.** The objective is an expectation whose *distribution* depends on $\theta$, so you cannot differentiate the integrand and be done. But $\nabla p = p\nabla\log p$ turns it into $\nabla E[R] = E[R\,\nabla\log\pi(a|s)]$ — **an expectation you can sample**. The line `grad = -p; grad[a] += 1` is exactly $\nabla\log\text{softmax}$, and everything else in `reinforce` is bookkeeping around it.

**Now the honest caveats, because this demo is a favourable case.** It is **tabular**: 64 independent parameters, no function approximation, no generalisation required. It is **on-policy**: every episode is used once and thrown away, which is why policy gradients are far more sample-hungry than Q-learning with replay. And the run is a **single seed** — REINFORCE is high-variance enough that another seed can look noticeably different, so the smoothness comparison would be more convincing averaged over five.

**The comparison against the oracle is what makes this cell evidence rather than a demonstration.** Session 2's exact value $V^*(\text{start}) = 0.642$ is the answer key; without it, a converged-looking curve tells you the algorithm stopped moving, not that it stopped in the right place. **Almost no RL experiment can perform this check** — and the habit of asking "converged to *what*?" is what this workshop is really teaching.

**Finally, note how little separates this from what trains modern language models.** PPO is REINFORCE plus a learned baseline (a critic) plus a trust region that refuses over-large updates. **Both additions are variance and stability armour on the estimator you just watched work**, and neither changes the underlying idea. Session 5 makes that translation explicit.

---
### 🕐 Session 5 of 5 — *The Road to RLHF* (~30 min)
**Goal:** connect this course to modern practice: reward models, KL anchors, and PPO's role.
**Builds on:** Session 4.

---

## 6. From Gridworld to Chatbots

The [LLM workshop](./LLMs_from_the_Ground_Up.ipynb) said 'preference tuning shapes judgment'; you now have the vocabulary for how:

1. **The policy** is the language model; a *state* is the prompt + text so far, an *action* is the next token, an *episode* is a completion.
2. **The reward** comes from a *reward model* trained on human preference pairs — [cross-entropy](../Intro_Math/Information_Theory/Information_Theory.ipynb) on 'which answer did the human prefer'.
3. **The optimizer** is a policy gradient with variance-reduction armor: PPO ≈ REINFORCE + a learned baseline (critic) + a *trust region* (clipped updates — don't move the policy further than the reward model's validity extends).
4. **The KL anchor**: reward is penalized by KL divergence from the pretrained model — 'improve preferences *without leaving the language manifold*'. DPO folds reward model + RL into one supervised loss on preference pairs, which is why it took over.

💡 **Intuition.** Everything hard about RLHF is Session 4's variance problem wearing a $10^{11}$-parameter costume, plus one new failure mode this course equips you to name: **reward hacking** — the policy exploiting the reward model where it's [off-distribution](./Uncertainty_in_ML.ipynb).

## 7. Conclusion

Bandits isolate exploration; Bellman makes value self-consistent; TD learns it from surprise (LMS's heartbeat again); policy gradients differentiate through luck with a baseline as armor; RLHF is all four at industrial scale. Every algorithm here was checked against an exact oracle — a habit worth keeping when the environments stop being 4×4.

---
## Where next

- [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb) — the policy being tuned.
- [Optimization](../Intro_Math/Optimization/Optimization.ipynb) — the SGD theory under the noise.
- [Beyond Kalman](../Intro_Time_Series/Beyond_Kalman.ipynb) — belief-state tracking: what 'state' means when you can't see it.